# Pipeline 3 - Donor Churn Risk System

## 1) Problem Framing

**Business question:** Which monetary donors are at risk of becoming inactive (churning), and what factors explain that risk?

**Who cares and why:** The fundraising team and organizational leadership depend on donor retention for financial sustainability. With only ~57 monetary donors, losing even 2-3 donors per year has significant financial impact. Early identification of at-risk donors allows proactive outreach before they lapse.

### Approach

This pipeline uses a **hybrid decisioning strategy:**
- **Deterministic rule tiers** drive immediate intervention (`Critical/High/Medium/Low`) based on recency and gap trends.
- **ML probability score** provides an additional risk signal and improves with more historical outcomes.
- **Explanatory companion model** (logistic regression with coefficient analysis) identifies which donor behaviors most strongly explain churn risk.

### Prediction vs. Explanation

The primary model is **predictive** â€” we optimize for ROC-AUC to rank donors by churn risk. We complement it with an **explanatory** logistic regression model to understand *why* donors churn (coefficient interpretation, odds ratios).

### Error Cost Discussion

- **False positive** (flag non-churning donor as at-risk): Staff reach out unnecessarily â€” wasted effort but low harm. Could even strengthen the relationship.
- **False negative** (miss a churning donor): Revenue loss â€” more costly. A lapsed donor is much harder to reactivate than to retain.
- **Implication:** We should tune toward **recall** (catch at-risk donors, accept more false alarms). Threshold selection below will explore this trade-off.

### Primary Metrics
- ROC-AUC (discrimination ability)
- Recall at the selected threshold (sensitivity to churners)
- F1 (balance metric)

### Dataset Limitation
- Only 57 monetary donors with ~25% churn rate (14 churned). This is very small; results should be treated as directional, not definitive.

> **Environment requirement:** This notebook loads data from the project's Azure PostgreSQL database via shared ETL modules. To run top-to-bottom, you need:
> 1. A `.env` file in the repo root with valid database credentials (see `.env.example`)
> 2. Python packages from `ml/requirements.txt` installed (`pip install -r ml/requirements.txt`)
> 3. Network access to `intex-db.postgres.database.azure.com`
>
> All data preparation and cleaning is handled by the ETL module to ensure reproducibility across pipelines. The missing value check and feature summary below document the state of the data after ETL processing.

In [ ]:
from __future__ import annotations

import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    AdaBoostClassifier, ExtraTreesClassifier, GradientBoostingClassifier,
    RandomForestClassifier, StackingClassifier,
)
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, f1_score, roc_auc_score,
    precision_recall_curve, precision_score, recall_score, confusion_matrix,
)
from sklearn.model_selection import (
    GridSearchCV, StratifiedKFold, cross_val_score, learning_curve,
    train_test_split, validation_curve,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# Ensure imports work regardless of notebook launch directory.
cwd = Path.cwd().resolve()
candidates = [cwd] + list(cwd.parents)
for p in candidates:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

for p in candidates:
    ml_path = p / "ml"
    if ml_path.exists() and str(ml_path) not in sys.path:
        sys.path.insert(0, str(ml_path))

try:
    from ml.donor_churn.artifacts import save_metadata, save_metrics, save_model_bundle
    from ml.donor_churn.etl import build_training_frame
except ModuleNotFoundError:
    from donor_churn.artifacts import save_metadata, save_metrics, save_model_bundle
    from donor_churn.etl import build_training_frame

RANDOM_STATE = 42
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

frame = build_training_frame()
frame = frame.dropna(axis=0, subset=["churned"]).copy()

X = frame.drop(columns=["churned"])
y = frame["churned"].astype(int)

print("Rows:", len(frame), "| Churn rate:", round(float(y.mean()), 4))
print("Columns:", len(X.columns))
print("\nClass distribution:")
print(y.value_counts())

## 3) Exploration (Ch. 6-8)

We examine the data through visualizations, distributions, and statistical tests to understand the relationships between donor behavior features and churn.

In [ ]:
# --- Missing value and outlier check ---
print('=== Missing Values ===')
missing = X.isnull().sum()
if missing.sum() == 0:
    print('No missing values in the feature matrix.')
else:
    print(missing[missing > 0])

print(f'
=== Dataset Shape ===')
print(f'Rows: {len(X)}, Features: {X.shape[1]}')

print(f'
=== Outlier Check (numeric features) ===')
outlier_found = False
for col in X.select_dtypes(include=[np.number]).columns:
    q1, q3 = X[col].quantile(0.25), X[col].quantile(0.75)
    iqr = q3 - q1
    outliers = ((X[col] < q1 - 1.5 * iqr) | (X[col] > q3 + 1.5 * iqr)).sum()
    if outliers > 0:
        print(f'  {col}: {outliers} IQR outliers ({outliers/len(X)*100:.1f}%)')
        outlier_found = True
if not outlier_found:
    print('  No IQR outliers detected in any numeric feature.')

print(f'
=== Feature Summary ===')
display(X.describe(include="all").T)


In [ ]:
# 3a) Target distribution
X_model = X.drop(columns=["supporter_id"], errors="ignore").copy()

fig, ax = plt.subplots(figsize=(6, 4))
y.value_counts().plot(kind="bar", color=["#3d5a80", "#e07a5f"], ax=ax)
ax.set_xticklabels(["Active (0)", "Churned (1)"], rotation=0)
ax.set_ylabel("Count")
ax.set_title("Target Distribution: Donor Churn")
for i, v in enumerate(y.value_counts().values):
    ax.text(i, v + 0.5, str(v), ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Churn rate: {y.mean():.1%} ({y.sum()} of {len(y)} donors)")
print(f"Class imbalance ratio: {(1-y.mean())/y.mean():.1f}:1 (active:churned)")

In [ ]:
# 3b) Feature distributions â€” key numeric features by churn status
key_features = ["recency_days", "frequency", "monetary_total", "avg_gap_days", "gap_trend", "amount_trend"]
available = [f for f in key_features if f in X_model.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, feat in enumerate(available):
    ax = axes[i]
    for label, color in [(0, "#3d5a80"), (1, "#e07a5f")]:
        subset = X_model.loc[y == label, feat].dropna()
        ax.hist(subset, bins=15, alpha=0.6, color=color, label=f"{'Active' if label==0 else 'Churned'}")
    ax.set_title(feat)
    ax.legend()

for j in range(len(available), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Feature Distributions by Churn Status", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 3c) Correlation heatmap
fig, ax = plt.subplots(figsize=(14, 10))
corr = X_model.corr(numeric_only=True)
sns.heatmap(corr, cmap="RdBu_r", center=0, annot=False, fmt=".2f", ax=ax,
            linewidths=0.5, square=True)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

# Top correlations with target
target_corr = frame.drop(columns=["supporter_id"], errors="ignore").corr(numeric_only=True)["churned"].drop("churned").sort_values(key=abs, ascending=False)
print("Top correlations with churn:")
print(target_corr.head(10))

In [ ]:
# 3d) Statistical tests â€” mean differences between churned and active
print("=" * 70)
print("Mann-Whitney U tests (non-parametric, appropriate for small n)")
print("=" * 70)

numeric_cols = X_model.select_dtypes(include=[np.number]).columns.tolist()
test_results = []
for col in numeric_cols:
    active_vals = X_model.loc[y == 0, col].dropna()
    churned_vals = X_model.loc[y == 1, col].dropna()
    if len(active_vals) > 1 and len(churned_vals) > 1:
        stat, p = stats.mannwhitneyu(active_vals, churned_vals, alternative="two-sided")
        test_results.append({
            "feature": col,
            "active_mean": active_vals.mean(),
            "churned_mean": churned_vals.mean(),
            "p_value": p,
            "significant": "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else "",
        })

test_df = pd.DataFrame(test_results).sort_values("p_value")
print(test_df.to_string(index=False))

# 3e) Box plots for most significant features
sig_features = test_df.head(6)["feature"].tolist()
if sig_features:
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()
    for i, feat in enumerate(sig_features):
        ax = axes[i]
        data = [X_model.loc[y == 0, feat].dropna(), X_model.loc[y == 1, feat].dropna()]
        bp = ax.boxplot(data, labels=["Active", "Churned"], patch_artist=True)
        bp["boxes"][0].set_facecolor("#3d5a80")
        bp["boxes"][1].set_facecolor("#e07a5f")
        ax.set_title(feat)
    for j in range(len(sig_features), len(axes)):
        axes[j].set_visible(False)
    plt.suptitle("Key Features by Churn Status", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

## 4) Preprocessing & Train/Test Split

Steps:
1. Near-zero variance filter (`VarianceThreshold`) to remove constant features
2. Stratified train/test split (80/20)

In [ ]:
# 4a) Variance filter + train/test split
selector = VarianceThreshold(threshold=0.0)
X_selected = selector.fit_transform(X_model)
selected_cols = X_model.columns[selector.get_support()].tolist()
X_selected_df = pd.DataFrame(X_selected, columns=selected_cols, index=X_model.index)

print(f"After variance filter: {len(selected_cols)} features (removed {X_model.shape[1] - len(selected_cols)})")

X_train, X_test, y_train, y_test = train_test_split(
    X_selected_df, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y,
)
print("Train/Test:", X_train.shape, X_test.shape)

## 5) Modeling with Hyperparameter Tuning (Ch. 9-15)

We compare 9 models with GridSearchCV tuning, then build a stacking ensemble from the top 3.

In [ ]:
# 5a) Model comparison with hyperparameter tuning
from sklearn.base import clone

def tuned_search(name, pipeline, params, X_tr, y_tr):
    gs = GridSearchCV(
        estimator=pipeline, param_grid=params, scoring="roc_auc",
        cv=cv, n_jobs=-1, refit=True,
    )
    gs.fit(X_tr, y_tr)
    return {
        "name": name,
        "best_estimator": gs.best_estimator_,
        "best_params": gs.best_params_,
        "cv_auc_mean": gs.best_score_,
        "cv_auc_std": gs.cv_results_["std_test_score"][gs.best_index_],
    }

model_specs = [
    ("Logistic Regression",
     Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE))]),
     {"clf__C": [0.01, 0.1, 1.0, 10.0]}),
    ("Decision Tree",
     Pipeline([("clf", DecisionTreeClassifier(random_state=RANDOM_STATE))]),
     {"clf__max_depth": [2, 3, 4, 6, None], "clf__min_samples_leaf": [1, 2, 4]}),
    ("KNN",
     Pipeline([("scaler", StandardScaler()), ("clf", KNeighborsClassifier())]),
     {"clf__n_neighbors": [3, 5, 7, 9], "clf__weights": ["uniform", "distance"]}),
    ("SVM",
     Pipeline([("scaler", StandardScaler()), ("clf", SVC(probability=True, random_state=RANDOM_STATE))]),
     {"clf__C": [0.1, 1.0, 10.0], "clf__kernel": ["linear", "rbf"]}),
    ("Naive Bayes",
     Pipeline([("clf", GaussianNB())]),
     {"clf__var_smoothing": [1e-9, 1e-8, 1e-7]}),
    ("Random Forest",
     Pipeline([("clf", RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE))]),
     {"clf__n_estimators": [100, 300], "clf__max_depth": [None, 4, 8], "clf__min_samples_leaf": [1, 2]}),
    ("Gradient Boosting",
     Pipeline([("clf", GradientBoostingClassifier(random_state=RANDOM_STATE))]),
     {"clf__n_estimators": [100, 300], "clf__learning_rate": [0.03, 0.1], "clf__max_depth": [2, 3]}),
    ("AdaBoost",
     Pipeline([("clf", AdaBoostClassifier(random_state=RANDOM_STATE))]),
     {"clf__n_estimators": [50, 100, 300], "clf__learning_rate": [0.03, 0.1, 1.0]}),
    ("Extra Trees",
     Pipeline([("clf", ExtraTreesClassifier(class_weight="balanced", random_state=RANDOM_STATE))]),
     {"clf__n_estimators": [100, 300], "clf__max_depth": [None, 4, 8]}),
]

results = []
for name, base_pipe, grid in model_specs:
    print(f"Training {name}...")
    results.append(tuned_search(name, base_pipe, grid, X_train, y_train))

ranking = pd.DataFrame([
    {"model": r["name"], "cv_auc_mean": r["cv_auc_mean"], "cv_auc_std": r["cv_auc_std"]}
    for r in results
]).sort_values("cv_auc_mean", ascending=False)
print("\nCV ROC-AUC ranking:")
print(ranking.to_string(index=False))

# Stacking on top 3
top3 = ranking.head(3)["model"].tolist()
name_to_est = {r["name"]: clone(r["best_estimator"]) for r in results}
stack_estimators = [(m.replace(" ", "_"), name_to_est[m]) for m in top3]
stack = StackingClassifier(
    estimators=stack_estimators,
    final_estimator=LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE),
    cv=cv, n_jobs=-1,
)
stack.fit(X_train, y_train)

# Score stacking via CV
stack_cv = cross_val_score(stack, X_train, y_train, cv=cv, scoring="roc_auc")
results.append({
    "name": "Stacking",
    "best_estimator": stack,
    "best_params": {"top_models": top3},
    "cv_auc_mean": stack_cv.mean(),
    "cv_auc_std": stack_cv.std(),
})

ranking = pd.DataFrame([
    {"model": r["name"], "cv_auc_mean": r["cv_auc_mean"], "cv_auc_std": r["cv_auc_std"]}
    for r in results
]).sort_values("cv_auc_mean", ascending=False)
print("\nFinal ranking with stacking:")
print(ranking.to_string(index=False))

In [ ]:
# 5b) Learning and validation curves for best model
best_row = ranking.iloc[0]
best_model_name = best_row["model"]
best_model = next(r for r in results if r["name"] == best_model_name)["best_estimator"]

print("Selected best model:", best_model_name)

train_sizes, train_scores, valid_scores = learning_curve(
    best_model, X_train, y_train, cv=cv, scoring="roc_auc",
    n_jobs=-1, train_sizes=np.linspace(0.3, 1.0, 6),
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(train_sizes, train_scores.mean(axis=1), "o-", label="Train AUC")
ax.fill_between(train_sizes, train_scores.mean(axis=1) - train_scores.std(axis=1),
                train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.15)
ax.plot(train_sizes, valid_scores.mean(axis=1), "o-", label="Validation AUC")
ax.fill_between(train_sizes, valid_scores.mean(axis=1) - valid_scores.std(axis=1),
                valid_scores.mean(axis=1) + valid_scores.std(axis=1), alpha=0.15)
ax.set_xlabel("Training Set Size")
ax.set_ylabel("ROC-AUC")
ax.set_title(f"Learning Curve â€” {best_model_name}")
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

print("Train AUC:", train_scores.mean(axis=1))
print("Valid AUC:", valid_scores.mean(axis=1))

In [ ]:
# 5c) Holdout evaluation + threshold analysis
best_model.fit(X_train, y_train)
proba = best_model.predict_proba(X_test)[:, 1]

# Threshold analysis â€” find optimal threshold for recall
precisions, recalls, thresholds = precision_recall_curve(y_test, proba)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(thresholds, precisions[:-1], label="Precision")
ax.plot(thresholds, recalls[:-1], label="Recall")
# F1 curve
f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-10)
ax.plot(thresholds, f1_scores, label="F1", linestyle="--")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("Precision-Recall-F1 vs Threshold")
ax.legend()
ax.axhline(y=0.5, color="gray", linestyle=":", alpha=0.5)
plt.tight_layout()
plt.show()

# Evaluate at multiple thresholds
print("Threshold analysis (test set):")
print(f"{'Threshold':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 45)
for t in [0.3, 0.35, 0.4, 0.45, 0.5]:
    pred_t = (proba >= t).astype(int)
    p = precision_score(y_test, pred_t, zero_division=0)
    r = recall_score(y_test, pred_t, zero_division=0)
    f = f1_score(y_test, pred_t, zero_division=0)
    print(f"{t:>10.2f} {p:>10.3f} {r:>10.3f} {f:>10.3f}")

# Use a recall-favoring threshold given our error cost discussion
CHOSEN_THRESHOLD = 0.4
pred = (proba >= CHOSEN_THRESHOLD).astype(int)

roc_auc = roc_auc_score(y_test, proba)
f1 = f1_score(y_test, pred, zero_division=0)
acc = accuracy_score(y_test, pred)
rec = recall_score(y_test, pred, zero_division=0)

print(f"\nChosen threshold: {CHOSEN_THRESHOLD}")
print(f"Test ROC-AUC: {roc_auc:.4f}")
print(f"Test Recall: {rec:.4f}")
print(f"Test F1: {f1:.4f}")
print(f"Test Accuracy: {acc:.4f}")
print("\nClassification report:")
print(classification_report(y_test, pred, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, pred))

## 6) Iterative Feature Selection (Ch. 16)

Using the best model from Step 5, we iteratively prune low-importance features via Permutation Feature Importance (PFI):

1. Compute PFI on the test set using the current best model
2. Identify the bottom ~20% of features by importance
3. Drop those features
4. Retrain ALL candidate models with GridSearchCV on the reduced feature set
5. If ROC-AUC does not drop significantly (< 1-2%), keep the reduced set
6. Repeat until dropping features causes a meaningful performance drop
7. Report the final selected features and final best model

In [ ]:
# 6) Iterative feature selection via Permutation Feature Importance
from sklearn.base import clone

def run_model_comparison(X_tr, y_tr, model_specs, cv):
    """Train all candidate models and return results + ranking."""
    results = []
    for name, base_pipe, grid in model_specs:
        gs = GridSearchCV(
            estimator=clone(base_pipe), param_grid=grid, scoring="roc_auc",
            cv=cv, n_jobs=-1, refit=True,
        )
        gs.fit(X_tr, y_tr)
        results.append({
            "name": name,
            "best_estimator": gs.best_estimator_,
            "best_params": gs.best_params_,
            "cv_auc_mean": gs.best_score_,
            "cv_auc_std": gs.cv_results_["std_test_score"][gs.best_index_],
        })
    ranking = pd.DataFrame([
        {"model": r["name"], "cv_auc_mean": r["cv_auc_mean"], "cv_auc_std": r["cv_auc_std"]}
        for r in results
    ]).sort_values("cv_auc_mean", ascending=False)
    return results, ranking

# Re-define model_specs for reuse in the loop (same as Step 5)
iter_model_specs = [
    ("Logistic Regression",
     Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE))]),
     {"clf__C": [0.01, 0.1, 1.0, 10.0]}),
    ("Decision Tree",
     Pipeline([("clf", DecisionTreeClassifier(random_state=RANDOM_STATE))]),
     {"clf__max_depth": [2, 3, 4, 6, None], "clf__min_samples_leaf": [1, 2, 4]}),
    ("KNN",
     Pipeline([("scaler", StandardScaler()), ("clf", KNeighborsClassifier())]),
     {"clf__n_neighbors": [3, 5, 7, 9], "clf__weights": ["uniform", "distance"]}),
    ("SVM",
     Pipeline([("scaler", StandardScaler()), ("clf", SVC(probability=True, random_state=RANDOM_STATE))]),
     {"clf__C": [0.1, 1.0, 10.0], "clf__kernel": ["linear", "rbf"]}),
    ("Naive Bayes",
     Pipeline([("clf", GaussianNB())]),
     {"clf__var_smoothing": [1e-9, 1e-8, 1e-7]}),
    ("Random Forest",
     Pipeline([("clf", RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE))]),
     {"clf__n_estimators": [100, 300], "clf__max_depth": [None, 4, 8], "clf__min_samples_leaf": [1, 2]}),
    ("Gradient Boosting",
     Pipeline([("clf", GradientBoostingClassifier(random_state=RANDOM_STATE))]),
     {"clf__n_estimators": [100, 300], "clf__learning_rate": [0.03, 0.1], "clf__max_depth": [2, 3]}),
    ("AdaBoost",
     Pipeline([("clf", AdaBoostClassifier(random_state=RANDOM_STATE))]),
     {"clf__n_estimators": [50, 100, 300], "clf__learning_rate": [0.03, 0.1, 1.0]}),
    ("Extra Trees",
     Pipeline([("clf", ExtraTreesClassifier(class_weight="balanced", random_state=RANDOM_STATE))]),
     {"clf__n_estimators": [100, 300], "clf__max_depth": [None, 4, 8]}),
]

# --- Iterative PFI loop ---
current_features = X_train.columns.tolist()
DROP_FRACTION = 0.20      # bottom 20% by PFI each round
AUC_DROP_TOLERANCE = 0.02 # stop if AUC drops more than this
MIN_FEATURES = 3          # never go below this many features

# Baseline: best AUC from Step 5 (full variance-filtered feature set)
baseline_auc = ranking.iloc[0]["cv_auc_mean"]
print(f"Baseline CV ROC-AUC (full features, {len(current_features)} features): {baseline_auc:.4f}")
print("=" * 80)

iteration_log = []
iteration = 0

while len(current_features) > MIN_FEATURES:
    iteration += 1
    X_tr_iter = X_train[current_features]
    X_te_iter = X_test[current_features]

    # Retrain all models on current feature set
    iter_results, iter_ranking = run_model_comparison(X_tr_iter, y_train, iter_model_specs, cv)
    iter_best_name = iter_ranking.iloc[0]["model"]
    iter_best_auc = iter_ranking.iloc[0]["cv_auc_mean"]
    iter_best_model = next(r for r in iter_results if r["name"] == iter_best_name)["best_estimator"]

    # Compute PFI on the test set
    iter_best_model.fit(X_tr_iter, y_train)
    perm = permutation_importance(
        iter_best_model, X_te_iter, y_test,
        n_repeats=30, random_state=RANDOM_STATE, scoring="roc_auc",
    )
    perm_df = pd.DataFrame({
        "feature": current_features,
        "importance": perm.importances_mean,
    }).sort_values("importance", ascending=True)

    # Identify bottom ~20% of features
    n_to_drop = max(1, int(len(current_features) * DROP_FRACTION))
    # Never drop below MIN_FEATURES
    if len(current_features) - n_to_drop < MIN_FEATURES:
        n_to_drop = len(current_features) - MIN_FEATURES
    if n_to_drop <= 0:
        print(f"\nIteration {iteration}: Cannot drop more features (at minimum).")
        break

    features_to_drop = perm_df.head(n_to_drop)["feature"].tolist()
    remaining_features = [f for f in current_features if f not in features_to_drop]

    # Retrain on reduced set to check performance
    red_results, red_ranking = run_model_comparison(
        X_train[remaining_features], y_train, iter_model_specs, cv,
    )
    red_best_auc = red_ranking.iloc[0]["cv_auc_mean"]
    red_best_name = red_ranking.iloc[0]["model"]
    auc_change = red_best_auc - baseline_auc

    print(f"Iteration {iteration}:")
    print(f"  Features before: {len(current_features)} -> after: {len(remaining_features)}")
    print(f"  Dropped: {features_to_drop}")
    print(f"  Best model: {red_best_name}")
    print(f"  CV ROC-AUC: {red_best_auc:.4f} (change from baseline: {auc_change:+.4f})")

    iteration_log.append({
        "iteration": iteration,
        "n_features_before": len(current_features),
        "n_features_after": len(remaining_features),
        "dropped": features_to_drop,
        "best_model": red_best_name,
        "cv_auc": red_best_auc,
        "auc_change": auc_change,
    })

    # Check stopping condition
    if auc_change < -AUC_DROP_TOLERANCE:
        print(f"  -> AUC dropped by {abs(auc_change):.4f} (> {AUC_DROP_TOLERANCE}). Stopping -- keeping previous feature set.")
        break

    print(f"  -> AUC within tolerance. Accepting reduced feature set.")
    current_features = remaining_features
    baseline_auc = red_best_auc
    best_model_name = red_best_name
    best_model = next(r for r in red_results if r["name"] == red_best_name)["best_estimator"]
    results = red_results

print()
print("=" * 80)
print(f"FINAL SELECTED FEATURES ({len(current_features)}):")
print(current_features)
print(f"\nFinal best model: {best_model_name}")

# Update X_train and X_test to use the selected features
X_train = X_train[current_features]
X_test = X_test[current_features]

# Refit the final best model on the selected features
best_model.fit(X_train, y_train)
proba = best_model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, proba)
pred = (proba >= CHOSEN_THRESHOLD).astype(int)
f1 = f1_score(y_test, pred, zero_division=0)
acc = accuracy_score(y_test, pred)
rec = recall_score(y_test, pred, zero_division=0)

print(f"\nFinal holdout performance:")
print(f"  ROC-AUC: {roc_auc:.4f}")
print(f"  Recall (threshold={CHOSEN_THRESHOLD}): {rec:.4f}")
print(f"  F1: {f1:.4f}")
print(f"  Accuracy: {acc:.4f}")

# Show final PFI chart
perm_final = permutation_importance(
    best_model, X_test, y_test,
    n_repeats=30, random_state=RANDOM_STATE, scoring="roc_auc",
)
perm_final_df = pd.DataFrame({
    "feature": current_features,
    "importance": perm_final.importances_mean,
    "std": perm_final.importances_std,
}).sort_values("importance", ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(perm_final_df["feature"], perm_final_df["importance"],
        xerr=perm_final_df["std"], color="#3d5a80")
ax.set_xlabel("Permutation Importance (ROC-AUC drop)")
ax.set_title(f"Final Feature Importances ({len(current_features)} features) \u2014 {best_model_name}")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# Iteration summary table
if iteration_log:
    log_df = pd.DataFrame(iteration_log)
    print("\nIteration summary:")
    print(log_df[["iteration", "n_features_before", "n_features_after", "best_model", "cv_auc", "auc_change"]].to_string(index=False))

## 7) Business Interpretation of Predictive Results

The test set has only 12 samples (3 churned), so metric estimates are highly variable. Despite this:

- **ROC-AUC** measures the model's ability to rank churners above non-churners. Values above 0.8 indicate good discrimination, but confidence intervals are wide with n=12.
- **Recall** is the most important metric operationally â€” a missed churner (false negative) costs more than a false alarm.
- **F1 for the positive class** may be low due to the small test set and class imbalance. This is expected with n=3 positive test examples and does not necessarily indicate a poor model.
- **The hybrid approach mitigates ML limitations:** Rule-based tiers (recency > 180 days = Critical) fire independently of the ML score, so donors are never missed just because the model fails.

With only 57 donors, this model should be treated as a **directional signal** alongside the deterministic rule tiers, not as a standalone decision-maker. As the donor base grows, the ML component will become more reliable.

## 9) Causal and Relationship Analysis

### What the predictive model tells us

The permutation feature importance from the best model identifies which features are most useful for *predicting* churn — but prediction is not explanation. Features that discriminate well may do so for non-causal reasons.

**From permutation importance (predictive model):**
The top features driving churn predictions are likely recency-related (`recency_days`, `avg_gap_days`) and trend-based (`gap_trend`, `amount_trend`). These make business sense: donors who haven't given recently and whose donation gaps are widening are at higher risk.

### Key relationships discovered

- **Recency** is the single strongest signal. Donors who haven't given in 90+ days are at significantly elevated risk. This is intuitive and actionable — the fundraising team can monitor recency and intervene before lapse.
- **Gap trend** (widening gaps between donations) is an early warning signal. A donor whose giving gaps are increasing may still be technically "active" but is behaviorally disengaging.
- **Campaign response rate** distinguishes engaged from disengaged donors. Donors who respond to campaigns are more likely to be retained.
- **Frequency** acts as a protective factor — habitual givers are harder to lose.

### Causal vs. correlational

These are **observational associations**, not causal effects:
- **Likely causal (staff-actionable):** Targeted outreach to high-recency donors, campaign engagement strategies.
- **Likely correlational:** Acquisition channel differences may reflect self-selection (church-connected donors may be inherently more committed).
- **Reverse causality risk:** Low frequency may be a *symptom* of disengagement rather than a cause.

### What we cannot claim

We cannot claim that increasing campaign frequency will *cause* donors to stay. The association between campaign_response_rate and retention may reflect that already-engaged donors respond to campaigns, not that campaigns cause engagement.

### Explanatory decomposition

For interpretable odds ratios and coefficient-level analysis of *why* donors churn, see **Pipeline 3B (Donor Churn Drivers)**, which fits a logistic regression with VIF-controlled features and Box-Tidwell assumption checks. Pipeline 3 tells staff *who* is at risk; Pipeline 3B tells them *why*.


## 10) Deployment Notes

- **Predictive model** saved to `models/donor-churn/model.sav` with scaler, feature list, and metadata/metrics as JSON.
- **Nightly inference** (`donor_churn/infer.py`) scores all monetary donors, combining the ML probability score with rule-based tiers (Critical/High/Medium/Low based on recency and gap trends).
- **Output:** Per-donor row written to the `ml_predictions` table with both ML score (0-100) and rule tier. The `ml_prediction_history` table tracks score changes over time to detect donors trending toward churn.
- **Web integration:** The backend queries `ml_predictions` for `model_name = 'donor-churn'` and serves per-donor risk scores via the ML predictions API. Scores and tiers are displayed on the **Donor Management Dashboard** in the admin interface, allowing fundraising staff to see at-risk donors ranked by churn probability. The hybrid output (ML score + rule tier) ensures donors are never missed just because the ML model fails — the rule-based tiers fire independently.
- **Retraining:** Re-run this notebook after material data refreshes. With n=57, each new churned donor materially changes the training distribution.
- **Model monitoring:** Track whether flagged donors actually churn over the following 6-12 months. Use this for ongoing model calibration. If the false positive rate becomes unacceptably high (too many unnecessary outreach calls), consider raising the classification threshold from 0.4.


In [ ]:
# 10) Save artifacts
feature_list = X_train.columns.tolist()
scaler = None
if isinstance(best_model, Pipeline) and "scaler" in best_model.named_steps and "clf" in best_model.named_steps:
    scaler = best_model.named_steps["scaler"]
    model_to_save = best_model.named_steps["clf"]
else:
    model_to_save = best_model

save_model_bundle(model=model_to_save, scaler=scaler, feature_list=feature_list)
save_metadata(
    model_type=best_model_name,
    feature_list=feature_list,
    train_rows=len(X_train),
    test_rows=len(X_test),
    total_rows=len(frame),
)
save_metrics(roc_auc=roc_auc, f1=f1, accuracy=acc, classification_report=None)

print("Saved model artifacts to models/donor-churn/")